In [1]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, roc_curve

sys.path.insert(0, "/kaggle/input/datasets/kabirsinghsachdeva/model-and-function")
from model import BaselineCNN
from losses import FocalLoss

# ---------------- config ----------------
CSV_PATH       = "/kaggle/input/datasets/kabirsinghsachdeva/manifest-2balanced/DR_all.csv"        # TODO: path to DR_all.csv
PROCESSED_ROOT = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/processed"         # TODO: folder holding the processed images

LOSS        = "bce"        # "bce" (control) or "focal"
IMG_SIZE    = 512
BATCH       = 32
EPOCHS      = 30
PATIENCE    = 5            # stop if val AUROC does not improve for this many epochs
LR, WD      = 1e-3, 1e-4
TARGET_SENS = 0.90         # threshold is chosen on VAL to reach this sensitivity
SEED        = 42
SAMPLE_RUN  = False         # True = 10% of train + val, for debugging

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device  = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
print("device:", device)

device: cuda


In [2]:
df = pd.read_csv(CSV_PATH)
df = df[df["split"] != "unused"].reset_index(drop=True)

index = {}
for dirpath, _, files in os.walk(PROCESSED_ROOT):
    for f in files:
        index[f] = os.path.join(dirpath, f)
df["proc_path"] = df["filename"].map(index)
assert df["proc_path"].notna().all(), df[df["proc_path"].isna()].head()

train_df = df[df["split"] == "train"]
val_df   = df[df["split"] == "val"]
if SAMPLE_RUN:
    train_df = train_df.sample(frac=0.10, random_state=SEED)
    val_df   = val_df.sample(frac=0.10, random_state=SEED)
print("train", len(train_df), "DR share", round(train_df["label"].mean(), 3))
print("val  ", len(val_df),   "DR share", round(val_df["label"].mean(), 3))

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tfm = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomAffine(degrees=15, scale=(0.9, 1.1)),          # mild rotation and zoom
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tfm = T.Compose([                                      # no random changes for val/test
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

class DRDataset(Dataset):
    def __init__(self, frame, tfm):
        self.paths  = frame["proc_path"].tolist()
        self.labels = frame["label"].values.astype("float32")
        self.tfm = tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), torch.tensor(self.labels[i])

train_loader = DataLoader(DRDataset(train_df, train_tfm), batch_size=BATCH, shuffle=True,
                          num_workers=4, pin_memory=use_amp, drop_last=True)
val_loader   = DataLoader(DRDataset(val_df, eval_tfm), batch_size=BATCH, shuffle=False,
                          num_workers=4, pin_memory=use_amp)

train 30287 DR share 0.501
val   6483 DR share 0.245


In [3]:
def get_loss(name):
    if name == "bce":
        return nn.BCEWithLogitsLoss()
    if name == "focal":
        return FocalLoss(gamma=2.0)
    raise ValueError(name)

model     = BaselineCNN().to(device)
criterion = get_loss(LOSS)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
scaler    = torch.amp.GradScaler("cuda", enabled=use_amp)

In [4]:
def threshold_for_sensitivity(y, p, target):
    """Highest threshold whose sensitivity is at least `target` (this gives the best specificity)."""
    fpr, tpr, thr = roc_curve(y, p)
    i = int(np.argmax(tpr >= target))
    return float(thr[i]), float(tpr[i]), float(1 - fpr[i])

def evaluate(loader):
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                logits = model(x).squeeze(1)
            all_logits.append(logits.float().cpu()); all_y.append(y)
    logits = torch.cat(all_logits); y = torch.cat(all_y)
    probs = torch.sigmoid(logits).numpy()
    loss = criterion(logits, y).item()
    return y.numpy(), probs, loss

history, best_auc, bad_epochs = [], -1.0, 0
CKPT = f"/kaggle/working/best_{LOSS}.pt" if os.path.exists("/kaggle/working") else f"best_{LOSS}.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    run_loss, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=use_amp):
            logits = model(x).squeeze(1)
        loss = criterion(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * len(y); n += len(y)

    yv, pv, val_loss = evaluate(val_loader)
    auc = roc_auc_score(yv, pv)
    thr, sens, spec = threshold_for_sensitivity(yv, pv, TARGET_SENS)
    history.append(dict(epoch=epoch, train_loss=run_loss / n, val_loss=val_loss,
                        val_auc=auc, val_sens_at_target=sens, val_spec_at_target=spec))
    print(f"epoch {epoch:2d} | train loss {run_loss / n:.4f} | val loss {val_loss:.4f} | "
          f"val AUROC {auc:.4f} | spec at {TARGET_SENS:.0%} sens {spec:.3f} | {time.time() - t0:.0f}s")

    if auc > best_auc:
        best_auc, bad_epochs = auc, 0
        torch.save(model.state_dict(), CKPT)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stop"); break

pd.DataFrame(history).to_csv(CKPT.replace(".pt", "_history.csv"), index=False)
print("best val AUROC:", round(best_auc, 4), "| saved", CKPT)

epoch  1 | train loss 0.6941 | val loss 0.6948 | val AUROC 0.5650 | spec at 90% sens 0.164 | 265s
epoch  2 | train loss 0.6864 | val loss 0.6679 | val AUROC 0.5784 | spec at 90% sens 0.166 | 257s
epoch  3 | train loss 0.6847 | val loss 0.6365 | val AUROC 0.5745 | spec at 90% sens 0.162 | 258s
epoch  4 | train loss 0.6829 | val loss 0.6609 | val AUROC 0.5963 | spec at 90% sens 0.171 | 257s
epoch  5 | train loss 0.6810 | val loss 0.6642 | val AUROC 0.5955 | spec at 90% sens 0.180 | 256s
epoch  6 | train loss 0.6797 | val loss 0.6930 | val AUROC 0.5940 | spec at 90% sens 0.164 | 254s
epoch  7 | train loss 0.6786 | val loss 0.6977 | val AUROC 0.5892 | spec at 90% sens 0.142 | 255s
epoch  8 | train loss 0.6768 | val loss 0.6683 | val AUROC 0.6193 | spec at 90% sens 0.161 | 254s
epoch  9 | train loss 0.6764 | val loss 0.6694 | val AUROC 0.6203 | spec at 90% sens 0.208 | 253s
epoch 10 | train loss 0.6742 | val loss 0.7088 | val AUROC 0.6220 | spec at 90% sens 0.195 | 254s
epoch 11 | train los

In [5]:
model.load_state_dict(torch.load(CKPT, map_location=device))
yv, pv, _ = evaluate(val_loader)
thr, sens, spec = threshold_for_sensitivity(yv, pv, TARGET_SENS)
print(f"VAL: AUROC {roc_auc_score(yv, pv):.4f} | threshold {thr:.4f} -> sensitivity {sens:.3f}, specificity {spec:.3f}")
json.dump(dict(loss=LOSS, threshold=thr, seed=SEED, img_size=IMG_SIZE, best_val_auc=float(best_auc)),
          open(CKPT.replace(".pt", "_threshold.json"), "w"))

VAL: AUROC 0.7509 | threshold 0.3448 -> sensitivity 0.900, specificity 0.301


In [6]:
RUN_TEST = False          # set True only for the final run

if RUN_TEST:
    test_df = df[df["split"] == "test"]
    test_loader = DataLoader(DRDataset(test_df, eval_tfm), batch_size=BATCH, shuffle=False, num_workers=4)
    yt, pt, _ = evaluate(test_loader)
    pred = (pt >= thr).astype(int)
    tp = int(((pred == 1) & (yt == 1)).sum()); fn = int(((pred == 0) & (yt == 1)).sum())
    tn = int(((pred == 0) & (yt == 0)).sum()); fp = int(((pred == 1) & (yt == 0)).sum())
    print(f"TEST AUROC {roc_auc_score(yt, pt):.4f}")
    print(f"sensitivity {tp / (tp + fn):.3f} | specificity {tn / (tn + fp):.3f} | precision {tp / max(tp + fp, 1):.3f}")
    print("confusion [[TN FP] [FN TP]]:", [[tn, fp], [fn, tp]])